# Week 10: RL and RLC Circuits -- Transient Response & Resonance
### PHASE 4: AC & Resonance

*Physics II (PHY102) . 3 Hours . Dr. Arif Solmaz*

## Learning Objectives

By the end of this week, you should be able to:

- Describe how an inductor opposes changes in current and define the RL time constant
- Analyze the transient (switch-on) response of a series RL circuit
- Write the differential equation for a series RLC circuit and identify its natural frequency
- Distinguish between underdamped, critically damped, and overdamped responses
- Explain resonance in a driven RLC circuit and locate the resonance frequency
- Define the quality factor $Q$ and relate it to bandwidth and energy dissipation
- Solve quantitative problems involving RL time constants, RLC oscillations, and resonance

## 🎯 Core Mastery Connection

This week you combine resistors, inductors, and capacitors into RLC circuits. Given a circuit configuration, you identify the governing differential equation, predict the natural frequency and damping behavior, and determine whether the system oscillates or decays. At resonance, you predict maximum current and voltage amplification. This is the full Configuration → Law → Equation → Prediction → Verify cycle applied to AC circuits.

> *"RLC combines all components. Predict natural frequency, damping, and resonance."*

> **From Physics I:** Before starting this week, make sure you are comfortable with:
> - Simple harmonic motion and resonance (PHY I Weeks 11–12): oscillation frequency, amplitude, and damping
> - Differential equations at a conceptual level, as encountered in PHY I oscillations ($m\ddot{x} + b\dot{x} + kx = 0$)
> - RC circuits from Week 06 of this course: time constants and exponential charging/discharging

In [ ]:
# ── Setup ──────────────────────────────────────────────
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Math
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox

plt.rcParams.update({'font.size': 13, 'figure.figsize': (9, 5)})
print('All imports loaded successfully.')

---
## 1. Inductors and Self-Inductance

An **inductor** is a coil of wire that stores energy in its magnetic field. When current through an inductor changes, it induces an EMF that opposes the change (Faraday's law applied to itself).

$$V_L = L\,\frac{dI}{dt}$$

where $L$ is the **inductance** measured in henrys (H).

Key properties:
- An inductor **opposes changes** in current (not current itself).
- At DC steady state, an ideal inductor acts like a short circuit (wire).
- Energy stored: $U = \frac{1}{2}LI^2$ (analogous to $\frac{1}{2}CV^2$ for a capacitor).

---
## 2. RL Circuit Step Response

Consider a series RL circuit: a battery $V_0$, a resistor $R$, and an inductor $L$ connected in series, with a switch.

When the switch is closed at $t = 0$:

$$V_0 = IR + L\frac{dI}{dt}$$

The solution is:

$$I(t) = \frac{V_0}{R}\left(1 - e^{-t/\tau}\right)$$

where the **RL time constant** is:

$$\tau = \frac{L}{R}$$

Physical interpretation:
- At $t = 0$: current is zero (inductor blocks sudden change).
- At $t = \tau$: current reaches about 63% of its final value.
- At $t \gg \tau$: current approaches $I_{\text{max}} = V_0/R$ (inductor acts as wire).
- Larger $L$ means slower rise; larger $R$ means faster rise.

### Interactive: RL Circuit Response

Adjust $R$ and $L$ to see how the current builds up over time when the switch is closed. The voltage source is $V_0 = 10$ V.

In [ ]:
def rl_response(R=10.0, L=0.1):
    V0 = 10.0  # V
    tau = L / R
    I_max = V0 / R

    t_end = max(5 * tau, 0.001)
    t = np.linspace(0, t_end, 500)
    I = I_max * (1 - np.exp(-t / tau))
    VR = I * R
    VL = V0 * np.exp(-t / tau)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Current plot
    ax1.plot(t * 1e3, I * 1e3, 'b-', lw=2, label='$I(t)$')
    ax1.axhline(I_max * 1e3, color='blue', ls='--', alpha=0.4, label=f'$I_{{max}}$ = {I_max*1e3:.1f} mA')
    ax1.axvline(tau * 1e3, color='gray', ls=':', alpha=0.5)
    ax1.plot(tau * 1e3, I_max * (1 - np.exp(-1)) * 1e3, 'ko', markersize=8)
    ax1.annotate(f'  t = τ = {tau*1e3:.2f} ms\n  I = 0.63 × I_max',
                 xy=(tau * 1e3, I_max * 0.63 * 1e3),
                 xytext=(tau * 1e3 + t_end * 0.2 * 1e3, I_max * 0.4 * 1e3),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=10)
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Current (mA)')
    ax1.set_title(f'RL Circuit Current  |  τ = L/R = {tau*1e3:.3f} ms')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, I_max * 1.15 * 1e3)

    # Voltage plot
    ax2.plot(t * 1e3, VR, 'r-', lw=2, label='$V_R(t)$')
    ax2.plot(t * 1e3, VL, 'g-', lw=2, label='$V_L(t)$')
    ax2.axhline(V0, color='gray', ls='--', alpha=0.3)
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Voltage (V)')
    ax2.set_title('Voltages across R and L')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, V0 * 1.1)

    plt.tight_layout()
    plt.show()

interact(rl_response,
         R=FloatSlider(min=1, max=100, step=1, value=10, description='R (Ω)',
                       style={'description_width': '50px'}),
         L=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description='L (H)',
                       style={'description_width': '50px'}));

---
## 3. RLC Circuit -- Natural Response

A series **RLC circuit** contains a resistor $R$, inductor $L$, and capacitor $C$. If the capacitor is initially charged to $V_0$ and the circuit is closed, the governing equation is:

$$L\frac{d^2q}{dt^2} + R\frac{dq}{dt} + \frac{q}{C} = 0$$

This is identical in form to a **damped harmonic oscillator** (mass-spring-dashpot system)!

The key parameters are:
- **Natural (resonant) frequency**: $\omega_0 = \frac{1}{\sqrt{LC}}$
- **Damping coefficient**: $\gamma = \frac{R}{2L}$

Three regimes:

| Regime | Condition | Behavior |
|--------|-----------|----------|
| **Underdamped** | $\gamma < \omega_0$ ($R < 2\sqrt{L/C}$) | Oscillations that decay exponentially |
| **Critically damped** | $\gamma = \omega_0$ ($R = 2\sqrt{L/C}$) | Fastest return to zero without oscillation |
| **Overdamped** | $\gamma > \omega_0$ ($R > 2\sqrt{L/C}$) | Slow exponential decay, no oscillation |

### Interactive: RLC Circuit Response

Adjust $R$, $L$, and $C$ to observe how the capacitor voltage evolves after the circuit is closed. Watch the transition between underdamped, critically damped, and overdamped behavior.

In [ ]:
def rlc_response(R=10.0, L=0.1, C_uF=100.0):
    C = C_uF * 1e-6
    V0 = 5.0  # initial capacitor voltage

    omega0 = 1 / np.sqrt(L * C)
    gamma = R / (2 * L)
    R_crit = 2 * np.sqrt(L / C)
    f0 = omega0 / (2 * np.pi)

    t_end = max(10 / gamma if gamma > 0.1 else 10 / omega0, 0.001)
    t = np.linspace(0, t_end, 2000)

    if gamma < omega0:  # underdamped
        omega_d = np.sqrt(omega0**2 - gamma**2)
        Vc = V0 * np.exp(-gamma * t) * (np.cos(omega_d * t) + (gamma / omega_d) * np.sin(omega_d * t))
        regime = 'UNDERDAMPED'
        color = 'blue'
    elif abs(gamma - omega0) / omega0 < 0.01:  # critically damped
        Vc = V0 * (1 + gamma * t) * np.exp(-gamma * t)
        regime = 'CRITICALLY DAMPED'
        color = 'green'
    else:  # overdamped
        s1 = -gamma + np.sqrt(gamma**2 - omega0**2)
        s2 = -gamma - np.sqrt(gamma**2 - omega0**2)
        A1 = V0 * s2 / (s2 - s1)
        A2 = -V0 * s1 / (s2 - s1)
        Vc = A1 * np.exp(s1 * t) + A2 * np.exp(s2 * t)
        regime = 'OVERDAMPED'
        color = 'red'

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(t * 1e3, Vc, color=color, lw=2, label=f'{regime}')
    ax.axhline(0, color='black', lw=0.5)

    if gamma < omega0:
        envelope = V0 * np.exp(-gamma * t)
        ax.plot(t * 1e3, envelope, 'k--', alpha=0.3, label='Envelope')
        ax.plot(t * 1e3, -envelope, 'k--', alpha=0.3)

    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Capacitor Voltage (V)')
    ax.set_title(f'RLC Natural Response  |  $\\omega_0$ = {omega0:.1f} rad/s  |  '
                 f'$\\gamma$ = {gamma:.1f} rad/s  |  $R_{{crit}}$ = {R_crit:.1f} Ω')
    ax.legend(fontsize=12, loc='upper right')
    ax.grid(True, alpha=0.3)

    info_text = (f'R = {R:.1f} Ω\nL = {L:.3f} H\nC = {C_uF:.0f} μF\n'
                 f'f₀ = {f0:.1f} Hz\nR_crit = {R_crit:.1f} Ω')
    ax.text(0.82, 0.55, info_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

interact(rlc_response,
         R=FloatSlider(min=1, max=200, step=1, value=10, description='R (Ω)',
                       style={'description_width': '60px'}),
         L=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description='L (H)',
                       style={'description_width': '60px'}),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)',
                          style={'description_width': '60px'}));

---
## 4. Animation: RLC Oscillation with Damping

The following animation shows the voltage across the capacitor in an underdamped RLC circuit evolving in real time. The envelope of the oscillation decays as $e^{-\gamma t}$.

In [ ]:
# Animation: underdamped RLC oscillation
L_anim = 0.1    # H
C_anim = 100e-6 # F
R_anim = 5.0    # Ohm (underdamped)
V0_anim = 5.0   # V

omega0_a = 1 / np.sqrt(L_anim * C_anim)
gamma_a = R_anim / (2 * L_anim)
omega_d_a = np.sqrt(omega0_a**2 - gamma_a**2)

t_anim_end = 8 / gamma_a
t_full_a = np.linspace(0, t_anim_end, 1000)
Vc_full = V0_anim * np.exp(-gamma_a * t_full_a) * (
    np.cos(omega_d_a * t_full_a) + (gamma_a / omega_d_a) * np.sin(omega_d_a * t_full_a))
env_full = V0_anim * np.exp(-gamma_a * t_full_a)

fig_rlc, ax_rlc = plt.subplots(figsize=(11, 5))
ax_rlc.set_xlim(0, t_anim_end * 1e3)
ax_rlc.set_ylim(-V0_anim * 1.1, V0_anim * 1.1)
ax_rlc.set_xlabel('Time (ms)')
ax_rlc.set_ylabel('Capacitor Voltage (V)')
ax_rlc.set_title(f'Underdamped RLC Oscillation  |  R={R_anim}Ω, L={L_anim}H, C={C_anim*1e6:.0f}μF')
ax_rlc.axhline(0, color='black', lw=0.5)
ax_rlc.grid(True, alpha=0.3)

# Pre-draw faint envelope
ax_rlc.plot(t_full_a * 1e3, env_full, 'k--', alpha=0.2)
ax_rlc.plot(t_full_a * 1e3, -env_full, 'k--', alpha=0.2)

line_rlc, = ax_rlc.plot([], [], 'b-', lw=2)
dot_rlc, = ax_rlc.plot([], [], 'ro', markersize=7)
time_text_rlc = ax_rlc.text(0.02, 0.95, '', transform=ax_rlc.transAxes, fontsize=11,
                             verticalalignment='top')

n_frames_rlc = 120

def init_rlc():
    line_rlc.set_data([], [])
    dot_rlc.set_data([], [])
    return line_rlc, dot_rlc

def animate_rlc(i):
    idx = int(i * len(t_full_a) / n_frames_rlc)
    line_rlc.set_data(t_full_a[:idx] * 1e3, Vc_full[:idx])
    if idx > 0:
        dot_rlc.set_data([t_full_a[idx-1] * 1e3], [Vc_full[idx-1]])
        time_text_rlc.set_text(f't = {t_full_a[idx-1]*1e3:.1f} ms')
    return line_rlc, dot_rlc, time_text_rlc

anim_rlc = FuncAnimation(fig_rlc, animate_rlc, init_func=init_rlc,
                          frames=n_frames_rlc, interval=50, blit=False)
plt.close(fig_rlc)
HTML(anim_rlc.to_jshtml())

---
## 5. Driven RLC Circuit and Resonance

When we drive a series RLC circuit with an AC source $V(t) = V_0 \sin(\omega t)$, the **steady-state** amplitude of the current depends on the driving frequency $\omega$.

The impedance of the series RLC circuit is:

$$Z = \sqrt{R^2 + \left(\omega L - \frac{1}{\omega C}\right)^2}$$

The current amplitude is:

$$I_0 = \frac{V_0}{Z}$$

**Resonance** occurs when $\omega L = 1/(\omega C)$, i.e., at:

$$\omega_0 = \frac{1}{\sqrt{LC}}, \qquad f_0 = \frac{1}{2\pi\sqrt{LC}}$$

At resonance:
- $Z_{\min} = R$ (purely resistive)
- $I_0$ is maximum: $I_{0,\max} = V_0/R$
- The inductor and capacitor voltages cancel each other.

### Interactive: Resonance Curve

Sweep the driving frequency and observe the current amplitude. Adjusting $R$ changes the width (sharpness) of the resonance peak.

In [ ]:
def resonance_curve(R=10.0, L=0.1, C_uF=100.0):
    C = C_uF * 1e-6
    V0 = 10.0  # V peak

    omega0 = 1 / np.sqrt(L * C)
    f0 = omega0 / (2 * np.pi)

    # Frequency range centered on resonance
    f_min = max(f0 / 5, 1)
    f_max = f0 * 5
    f = np.linspace(f_min, f_max, 1000)
    omega = 2 * np.pi * f

    Z = np.sqrt(R**2 + (omega * L - 1 / (omega * C))**2)
    I0 = V0 / Z
    I0_max = V0 / R

    # Bandwidth (FWHM): Δf = R/(2πL)
    delta_f = R / (2 * np.pi * L)
    Q = omega0 * L / R

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Linear scale
    ax1.plot(f, I0 * 1e3, 'b-', lw=2)
    ax1.axvline(f0, color='red', ls='--', alpha=0.5, label=f'$f_0$ = {f0:.1f} Hz')
    ax1.axhline(I0_max * 1e3 / np.sqrt(2), color='gray', ls=':', alpha=0.5,
                label=f'$I_{{max}}/\\sqrt{{2}}$ (half-power)')

    # Shade bandwidth
    f_low = f0 - delta_f / 2
    f_high = f0 + delta_f / 2
    mask = (f >= f_low) & (f <= f_high)
    ax1.fill_between(f[mask], 0, I0[mask] * 1e3, alpha=0.15, color='red',
                      label=f'Bandwidth Δf = {delta_f:.1f} Hz')

    ax1.set_xlabel('Frequency (Hz)')
    ax1.set_ylabel('Current Amplitude (mA)')
    ax1.set_title(f'Resonance Curve  |  Q = {Q:.1f}')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, I0_max * 1.15 * 1e3)

    # Phase plot
    phi = np.arctan2(omega * L - 1 / (omega * C), R)
    ax2.plot(f, np.degrees(phi), 'g-', lw=2)
    ax2.axvline(f0, color='red', ls='--', alpha=0.5)
    ax2.axhline(0, color='black', lw=0.5)
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Impedance Phase Angle (degrees)')
    ax2.set_title('Impedance Phase Angle vs Frequency')
    ax2.set_ylim(-95, 95)
    ax2.grid(True, alpha=0.3)
    ax2.text(f0 * 1.1, 5, f'φ = 0 at $f_0$', fontsize=10, color='red')

    plt.tight_layout()
    plt.show()

interact(resonance_curve,
         R=FloatSlider(min=1, max=100, step=1, value=10, description='R (Ω)',
                       style={'description_width': '60px'}),
         L=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description='L (H)',
                       style={'description_width': '60px'}),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)',
                          style={'description_width': '60px'}));

---
## 6. Quality Factor ($Q$)

The **quality factor** $Q$ measures how "sharp" or "selective" a resonance is:

$$Q = \frac{\omega_0 L}{R} = \frac{1}{R}\sqrt{\frac{L}{C}}$$

Equivalent definitions:
- $Q = \frac{f_0}{\Delta f}$ where $\Delta f$ is the bandwidth (frequency width at half-power, $-3$ dB)
- $Q = 2\pi \times \frac{\text{energy stored}}{\text{energy dissipated per cycle}}$

Key relationships:
- High $Q$ = narrow resonance = low damping = long-lived oscillations
- Low $Q$ = broad resonance = high damping = quickly decaying oscillations
- $Q > 0.5$: underdamped; $Q = 0.5$: critically damped; $Q < 0.5$: overdamped

### Interactive: Q Factor Explorer

See how the quality factor $Q$ connects the sharpness of the resonance peak, the bandwidth, and the damping rate. Multiple curves with different $Q$ values are shown for comparison.

In [ ]:
def q_factor_explorer(L=0.1, C_uF=100.0):
    C = C_uF * 1e-6
    omega0 = 1 / np.sqrt(L * C)
    f0 = omega0 / (2 * np.pi)

    # Several R values for different Q
    R_values = [5, 10, 20, 50, 100]
    colors = ['darkblue', 'blue', 'green', 'orange', 'red']

    f_min = max(f0 / 4, 1)
    f_max = f0 * 4
    f = np.linspace(f_min, f_max, 1000)
    omega = 2 * np.pi * f

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    for R, c in zip(R_values, colors):
        Q = omega0 * L / R
        Z = np.sqrt(R**2 + (omega * L - 1 / (omega * C))**2)
        # Normalized current (I/I_max at resonance)
        I_norm = R / Z
        delta_f = f0 / Q if Q > 0 else np.inf

        ax1.plot(f / f0, I_norm, color=c, lw=2,
                 label=f'R={R}Ω, Q={Q:.1f}, Δf={delta_f:.1f} Hz')

    ax1.axhline(1 / np.sqrt(2), color='gray', ls=':', alpha=0.5, label='Half-power level')
    ax1.set_xlabel('$f / f_0$ (normalized frequency)')
    ax1.set_ylabel('$I / I_{max}$ (normalized current)')
    ax1.set_title(f'Resonance Curves for Different Q  |  $f_0$ = {f0:.1f} Hz')
    ax1.legend(fontsize=8, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.1)

    # Right panel: Q vs R
    R_range = np.linspace(1, 150, 200)
    Q_range = omega0 * L / R_range
    bw_range = f0 / Q_range

    ax2.plot(R_range, Q_range, 'b-', lw=2, label='Q factor')
    ax2_twin = ax2.twinx()
    ax2_twin.plot(R_range, bw_range, 'r-', lw=2, label='Bandwidth')

    ax2.set_xlabel('Resistance R (Ω)')
    ax2.set_ylabel('Q Factor', color='blue')
    ax2_twin.set_ylabel('Bandwidth Δf (Hz)', color='red')
    ax2.set_title('Q Factor and Bandwidth vs R')
    ax2.axhline(0.5, color='green', ls='--', alpha=0.5)
    ax2.text(120, 0.7, 'Q = 0.5\n(critical)', fontsize=9, color='green')
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2_twin.tick_params(axis='y', labelcolor='red')

    plt.tight_layout()
    plt.show()

interact(q_factor_explorer,
         L=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.1, description='L (H)',
                       style={'description_width': '60px'}),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)',
                          style={'description_width': '60px'}));

---
## 7. Mechanical Analogy

The series RLC circuit is mathematically identical to a damped mass-spring system:

| Mechanical | Electrical |
|------------|------------|
| Mass $m$ | Inductance $L$ |
| Damping constant $b$ | Resistance $R$ |
| Spring constant $k$ | $1/C$ |
| Displacement $x$ | Charge $q$ |
| Velocity $v$ | Current $I$ |
| Applied force $F$ | Driving voltage $V$ |
| $\omega_0 = \sqrt{k/m}$ | $\omega_0 = 1/\sqrt{LC}$ |

This analogy is extremely useful: any intuition you have about mass-spring systems transfers directly to RLC circuits!

---
## 8. Worked Examples

### Example 1: RL Time Constant

An RL circuit has $R = 50\,\Omega$ and $L = 0.2$ H, connected to a 12 V battery. Find:
(a) The time constant  
(b) The maximum (steady-state) current  
(c) The current at $t = 2$ ms  
(d) The time to reach 90% of maximum current

In [ ]:
# Worked Example 1: RL time constant
R_w1 = 50     # Ohm
L_w1 = 0.2    # H
V0_w1 = 12    # V

tau_w1 = L_w1 / R_w1
I_max_w1 = V0_w1 / R_w1
t_w1 = 2e-3   # s
I_at_t = I_max_w1 * (1 - np.exp(-t_w1 / tau_w1))
# 90%: 0.9 = 1 - exp(-t/tau) => t = -tau * ln(0.1)
t_90 = -tau_w1 * np.log(0.1)

print("Worked Example 1: RL Time Constant")
print("=" * 48)
print(f"Given:  R = {R_w1} Ω, L = {L_w1} H, V₀ = {V0_w1} V")
print(f"\n(a) Time constant:")
print(f"    τ = L/R = {L_w1}/{R_w1} = {tau_w1*1e3:.1f} ms")
print(f"\n(b) Maximum current:")
print(f"    I_max = V₀/R = {V0_w1}/{R_w1} = {I_max_w1:.3f} A = {I_max_w1*1e3:.0f} mA")
print(f"\n(c) Current at t = {t_w1*1e3:.0f} ms:")
print(f"    I = I_max × (1 - e^(-t/τ))")
print(f"    I = {I_max_w1:.3f} × (1 - e^(-{t_w1*1e3:.0f}/{tau_w1*1e3:.1f}))")
print(f"    I = {I_max_w1:.3f} × (1 - {np.exp(-t_w1/tau_w1):.4f})")
print(f"    I = {I_at_t:.4f} A = {I_at_t*1e3:.2f} mA")
print(f"\n(d) Time to reach 90% of I_max:")
print(f"    0.9 = 1 - e^(-t/τ)  →  t = -τ ln(0.1)")
print(f"    t = {tau_w1*1e3:.1f} × {-np.log(0.1):.4f}")
print(f"    t = {t_90*1e3:.2f} ms ≈ 2.3τ")

### Example 2: RLC Resonance Frequency

A series RLC circuit has $L = 50$ mH and $C = 20\,\mu$F. Find:
(a) The resonant frequency $f_0$  
(b) The impedance at resonance if $R = 25\,\Omega$  
(c) The maximum current if $V_0 = 10$ V  
(d) The voltage across the capacitor at resonance

In [ ]:
# Worked Example 2: RLC resonance frequency
L_w2 = 50e-3   # H
C_w2 = 20e-6   # F
R_w2 = 25       # Ohm
V0_w2 = 10      # V

omega0_w2 = 1 / np.sqrt(L_w2 * C_w2)
f0_w2 = omega0_w2 / (2 * np.pi)
Z_res = R_w2  # at resonance
I_max_w2 = V0_w2 / Z_res
# Voltage across C at resonance: V_C = I * X_C = I / (omega0 * C)
X_C = 1 / (omega0_w2 * C_w2)
V_C_res = I_max_w2 * X_C
Q_w2 = omega0_w2 * L_w2 / R_w2

print("Worked Example 2: RLC Resonance Frequency")
print("=" * 50)
print(f"Given:  L = {L_w2*1e3:.0f} mH, C = {C_w2*1e6:.0f} μF, R = {R_w2} Ω, V₀ = {V0_w2} V")
print(f"\n(a) Resonant frequency:")
print(f"    ω₀ = 1/√(LC) = 1/√({L_w2} × {C_w2})")
print(f"    ω₀ = {omega0_w2:.2f} rad/s")
print(f"    f₀ = ω₀/(2π) = {f0_w2:.2f} Hz")
print(f"\n(b) Impedance at resonance:")
print(f"    Z = R = {Z_res} Ω  (X_L and X_C cancel!)")
print(f"\n(c) Maximum current at resonance:")
print(f"    I_max = V₀/R = {V0_w2}/{R_w2} = {I_max_w2:.2f} A")
print(f"\n(d) Voltage across capacitor at resonance:")
print(f"    X_C = 1/(ω₀C) = {X_C:.2f} Ω")
print(f"    V_C = I_max × X_C = {I_max_w2:.2f} × {X_C:.2f} = {V_C_res:.2f} V")
print(f"    Note: V_C = Q × V₀ = {Q_w2:.2f} × {V0_w2} = {Q_w2*V0_w2:.2f} V  ✓")
print(f"\n    Q factor: Q = ω₀L/R = {Q_w2:.2f}")
print(f"    The capacitor voltage is {Q_w2:.1f}× larger than the source voltage!")

### Example 3: Q Factor Calculation

A radio receiver's tuning circuit has $L = 1.0$ mH and $C = 25$ pF, with a coil resistance of $R = 5.0\,\Omega$. Find:
(a) The resonant frequency  
(b) The Q factor  
(c) The bandwidth

In [ ]:
# Worked Example 3: Q factor calculation
L_w3 = 1.0e-3   # H
C_w3 = 25e-12    # F
R_w3 = 5.0       # Ohm

omega0_w3 = 1 / np.sqrt(L_w3 * C_w3)
f0_w3 = omega0_w3 / (2 * np.pi)
Q_w3 = omega0_w3 * L_w3 / R_w3
delta_f_w3 = f0_w3 / Q_w3

print("Worked Example 3: Q Factor Calculation")
print("=" * 48)
print(f"Given:  L = {L_w3*1e3:.1f} mH, C = {C_w3*1e12:.0f} pF, R = {R_w3} Ω")
print(f"\n(a) Resonant frequency:")
print(f"    ω₀ = 1/√(LC) = 1/√({L_w3} × {C_w3:.2e})")
print(f"    ω₀ = {omega0_w3:.2e} rad/s")
print(f"    f₀ = {f0_w3:.2e} Hz = {f0_w3/1e6:.3f} MHz")
print(f"\n(b) Q factor:")
print(f"    Q = ω₀L/R = {omega0_w3:.2e} × {L_w3}/{R_w3}")
print(f"    Q = {Q_w3:.1f}")
print(f"\n(c) Bandwidth:")
print(f"    Δf = f₀/Q = {f0_w3/1e6:.3f} MHz / {Q_w3:.1f}")
print(f"    Δf = {delta_f_w3:.0f} Hz = {delta_f_w3/1e3:.2f} kHz")
print(f"\n    This circuit can select a radio station at {f0_w3/1e6:.3f} MHz")
print(f"    while rejecting stations more than ±{delta_f_w3/2/1e3:.1f} kHz away.")

---
## 9. Comparing All Three Damping Regimes

The following plot compares underdamped, critically damped, and overdamped responses side by side for the same $L$ and $C$, with different $R$ values.

In [ ]:
L_cmp = 0.1     # H
C_cmp = 100e-6  # F
V0_cmp = 5.0
omega0_cmp = 1 / np.sqrt(L_cmp * C_cmp)
R_crit_cmp = 2 * np.sqrt(L_cmp / C_cmp)

R_under = R_crit_cmp * 0.2
R_crit = R_crit_cmp
R_over = R_crit_cmp * 3.0

t_cmp = np.linspace(0, 0.15, 2000)

fig, ax = plt.subplots(figsize=(11, 5))

for R_val, label, color, ls in [
    (R_under, f'Underdamped (R={R_under:.1f}Ω)', 'blue', '-'),
    (R_crit, f'Critically damped (R={R_crit:.1f}Ω)', 'green', '-'),
    (R_over, f'Overdamped (R={R_over:.1f}Ω)', 'red', '-')
]:
    gamma_v = R_val / (2 * L_cmp)
    if gamma_v < omega0_cmp * 0.99:
        wd = np.sqrt(omega0_cmp**2 - gamma_v**2)
        Vc = V0_cmp * np.exp(-gamma_v * t_cmp) * (
            np.cos(wd * t_cmp) + (gamma_v / wd) * np.sin(wd * t_cmp))
    elif gamma_v < omega0_cmp * 1.01:
        Vc = V0_cmp * (1 + gamma_v * t_cmp) * np.exp(-gamma_v * t_cmp)
    else:
        s1 = -gamma_v + np.sqrt(gamma_v**2 - omega0_cmp**2)
        s2 = -gamma_v - np.sqrt(gamma_v**2 - omega0_cmp**2)
        A1 = V0_cmp * s2 / (s2 - s1)
        A2 = -V0_cmp * s1 / (s2 - s1)
        Vc = A1 * np.exp(s1 * t_cmp) + A2 * np.exp(s2 * t_cmp)

    ax.plot(t_cmp * 1e3, Vc, color=color, ls=ls, lw=2, label=label)

ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Capacitor Voltage (V)')
ax.set_title(f'Three Damping Regimes  |  L={L_cmp*1e3:.0f} mH, C={C_cmp*1e6:.0f} μF, '
             f'$R_{{crit}}$ = {R_crit_cmp:.1f} Ω')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Summary

| Concept | Key Equation | Meaning |
|---------|-------------|----------|
| Inductor voltage | $V_L = L\,dI/dt$ | Opposes changes in current |
| RL time constant | $\tau = L/R$ | Time scale for current buildup |
| RL step response | $I(t) = (V_0/R)(1 - e^{-t/\tau})$ | Exponential approach to steady state |
| Natural frequency | $\omega_0 = 1/\sqrt{LC}$ | Oscillation frequency of undamped RLC |
| Damping coefficient | $\gamma = R/(2L)$ | Rate of energy loss |
| Critical resistance | $R_{\text{crit}} = 2\sqrt{L/C}$ | Boundary between oscillating and non-oscillating |
| Quality factor | $Q = \omega_0 L/R = f_0/\Delta f$ | Sharpness of resonance |
| Resonance impedance | $Z = R$ (minimum) | Maximum current at $\omega_0$ |

---
## Problem Set

> **Core Mastery Approach -- For each problem: Identify the configuration → Choose the law → Write the equation → Predict → Verify**

Work through the following problems on RL circuits, RLC circuits, resonance, and quality factor. Problems are graded by difficulty:

- **L1 (Basic):** Single-concept, direct application of one formula
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

Show all work, include units at every step, and verify that your answers are physically reasonable.

### L1 Problems (Basic)

**P1.** An RL circuit has $R = 120\;\Omega$ and $L = 0.60$ H, connected to a 24 V battery. Calculate the time constant and the maximum (steady-state) current.

<details><summary>Answer</summary>τ = L/R = 0.60/120 = 5.0×10⁻³ s = 5.0 ms; I_max = V/R = 24/120 = 0.20 A = 200 mA</details>

In [ ]:
# ✏️ [P1] Your solution here


**P2.** A series RLC circuit has $L = 20$ mH and $C = 50\;\mu$F. What is the resonant frequency $f_0$?

<details><summary>Answer</summary>ω₀ = 1/√(LC) = 1/√(0.020×50×10⁻⁶) = 1/√(10⁻⁶) = 1000 rad/s; f₀ = ω₀/(2π) = 1000/(2π) = 159 Hz</details>

In [ ]:
# ✏️ [P2] Your solution here


**P3.** An inductor stores 0.25 J of energy when carrying a current of 5.0 A. What is its inductance?

<details><summary>Answer</summary>U = ½LI² → L = 2U/I² = 2(0.25)/(5.0)² = 0.50/25 = 0.020 H = 20 mH</details>

In [ ]:
# ✏️ [P3] Your solution here


**P4.** In a series RLC circuit, $R = 50\;\Omega$, $L = 0.10$ H, and $C = 40\;\mu$F. What is the quality factor $Q$?

<details><summary>Answer</summary>ω₀ = 1/√(LC) = 1/√(0.10×40×10⁻⁶) = 1/√(4×10⁻⁶) = 500 rad/s; Q = ω₀L/R = (500)(0.10)/50 = 1.0</details>

In [ ]:
# ✏️ [P4] Your solution here


---
### L2 Problems (Intermediate)

**P5.** An RL circuit ($R = 200\;\Omega$, $L = 0.50$ H) is connected to a 40 V battery at $t = 0$. Find: (a) the current at $t = 1.0$ ms, (b) the voltage across the inductor at $t = 1.0$ ms, and (c) the time at which the energy stored in the inductor equals half of its maximum value.

<details><summary>Answer</summary>τ = L/R = 0.50/200 = 2.5 ms; I_max = 40/200 = 0.20 A; (a) I(1 ms) = 0.20(1-e^(-1/2.5)) = 0.20(1-0.6703) = 0.20×0.3297 = 0.0659 A = 65.9 mA; (b) V_L = V₀ e^(-t/τ) = 40 e^(-0.4) = 40×0.6703 = 26.8 V; (c) U = ½LI², U_max = ½LI_max². U = U_max/2 when I = I_max/√2 → 1-e^(-t/τ) = 1/√2 → e^(-t/τ) = 1-0.7071 = 0.2929 → t = -τ ln(0.2929) = 2.5×1.228 = 3.07 ms</details>

In [ ]:
# ✏️ [P5] Your solution here


**P6.** A series RLC circuit has $R = 10\;\Omega$, $L = 25$ mH, and $C = 100\;\mu$F. A capacitor initially charged to 12 V is connected. (a) Is the circuit underdamped, critically damped, or overdamped? (b) What is the oscillation frequency (if applicable)? (c) What is the initial energy stored and in how many oscillation cycles will it fall to $1/e$ of this value?

<details><summary>Answer</summary>ω₀ = 1/√(0.025×10⁻⁴) = 1/√(2.5×10⁻⁶) = 632 rad/s; γ = R/(2L) = 10/(2×0.025) = 200 rad/s; R_crit = 2√(L/C) = 2√(0.025/10⁻⁴) = 2√250 = 31.6 Ω. (a) R=10 < 31.6 → underdamped; (b) ω_d = √(ω₀²-γ²) = √(632²-200²) = √(399424-40000) = √359424 = 600 rad/s, f_d = 95.5 Hz; (c) U₀ = ½CV² = ½(10⁻⁴)(144) = 7.2 mJ. Energy ∝ e^(-2γt), so it drops to 1/e when 2γt=1, t = 1/(2×200) = 2.5 ms. Number of cycles = t×f_d = 0.0025×95.5 ≈ 0.24 cycles (energy decays fast for Q≈1.6)</details>

In [ ]:
# ✏️ [P6] Your solution here


**P7.** A driven series RLC circuit with $R = 8.0\;\Omega$, $L = 50$ mH, and $C = 200\;\mu$F is connected to an AC source with peak voltage $V_0 = 20$ V. Find: (a) the resonant frequency, (b) the impedance and peak current at resonance, (c) the peak voltage across the capacitor at resonance.

<details><summary>Answer</summary>(a) ω₀ = 1/√(0.050×2×10⁻⁴) = 1/√(10⁻⁵) = 316 rad/s, f₀ = 50.3 Hz; (b) At resonance Z = R = 8.0 Ω; I₀ = V₀/R = 20/8.0 = 2.5 A; (c) X_C = 1/(ω₀C) = 1/(316×2×10⁻⁴) = 15.8 Ω; V_C = I₀X_C = 2.5×15.8 = 39.5 V; Note: V_C > V_source since Q = ω₀L/R = 316×0.050/8 = 1.98</details>

In [ ]:
# ✏️ [P7] Your solution here


**P8.** A radio tuning circuit uses a fixed inductor $L = 2.0$ mH and a variable capacitor. (a) What capacitance is needed to tune to a station at 550 kHz (AM band)? (b) What capacitance for 1600 kHz? (c) If the coil resistance is $R = 10\;\Omega$, what is the bandwidth at 550 kHz?

<details><summary>Answer</summary>(a) ω = 2π(550×10³) = 3.456×10⁶ rad/s; C = 1/(ω²L) = 1/[(3.456×10⁶)²×2×10⁻³] = 1/(2.389×10¹³×2×10⁻³) = 4.18×10⁻¹¹ F ≈ 42 pF; (b) ω = 2π(1.6×10⁶) = 1.005×10⁷ rad/s; C = 1/[(1.005×10⁷)²×2×10⁻³] = 4.95×10⁻¹² F ≈ 4.95 pF; (c) Δf = R/(2πL) = 10/(2π×2×10⁻³) = 796 Hz</details>

In [ ]:
# ✏️ [P8] Your solution here


---
### L3 Problems (Challenge)

**P9.** An engineer is designing an RL relay circuit. The relay coil has $L = 0.30$ H and $R = 15\;\Omega$, and the relay activates when the current reaches 0.80 A. The supply voltage is 18 V. (a) What is the steady-state current? (b) How long after the switch is closed does the relay activate? (c) When the switch is opened, the current must decay through a protective diode. How long does it take for the current to drop from 0.80 A to 0.05 A?

<details><summary>Answer</summary>τ = L/R = 0.30/15 = 0.020 s = 20 ms; I_max = V/R = 18/15 = 1.20 A; (a) 1.20 A; (b) 0.80 = 1.20(1-e^(-t/0.020)) → e^(-t/0.020) = 1-0.667 = 0.333 → t = -0.020×ln(0.333) = 0.020×1.099 = 22.0 ms; (c) Decay: I = 0.80 e^(-t/0.020), 0.05 = 0.80 e^(-t/0.020) → e^(-t/0.020) = 0.0625 → t = -0.020×ln(0.0625) = 0.020×2.773 = 55.5 ms</details>

In [ ]:
# ✏️ [P9] Your solution here


**P10.** A wireless power transfer system operates at the resonant frequency of its receiver circuit, which has $L = 10\;\mu$H, $C = 100$ pF, and coil resistance $R = 0.50\;\Omega$. (a) What is the operating frequency? (b) What is the quality factor? (c) If the induced EMF amplitude in the receiver coil is 0.50 V, what is the voltage across the capacitor at resonance? (d) What is the average power dissipated in the coil resistance?

<details><summary>Answer</summary>(a) ω₀ = 1/√(10×10⁻⁶×100×10⁻¹²) = 1/√(10⁻¹⁵) = 1/(3.162×10⁻⁸) = 3.162×10⁷ rad/s; f₀ = 5.03 MHz; (b) Q = ω₀L/R = (3.162×10⁷)(10⁻⁵)/0.50 = 632; (c) V_C = Q×V_source = 632×0.50 = 316 V; (d) I₀ = V_source/R = 0.50/0.50 = 1.0 A; P_avg = ½I₀²R = ½(1.0)²(0.50) = 0.25 W</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## Bridge to Next Week

We have seen how inductors and capacitors interact in transient and resonant circuits. Next week, we bring together everything we have learned about electric and magnetic fields into a unified framework: **Maxwell's Equations and Electromagnetic Waves**.

- **Displacement current**: Maxwell's crucial addition to Ampere's law -- a changing electric field acts as a source of magnetic field
- **The four Maxwell equations**: the complete, self-consistent set of laws governing all electromagnetic phenomena
- **Electromagnetic wave derivation**: how oscillating electric and magnetic fields sustain each other and propagate through space
- **Speed of light**: $c = 1/\sqrt{\mu_0 \varepsilon_0}$ -- emerging naturally from the theory
- **The electromagnetic spectrum**: from radio waves to gamma rays, all described by the same physics

Maxwell's equations are one of the greatest achievements of classical physics, unifying electricity, magnetism, and optics into a single elegant framework.